# Execute and inspect MongoDB queries

**Goal:** run a query, compare execution with profiling, and handle a workload's results.

Start Docker and run all cells in order with the BONDI Python environment.
The first code cell creates the named `notebook-workload` deployment and loads the
eight fictional businesses into `demo.business`.
No external data or connection configuration is needed. If a port is occupied,
change the port arguments in the setup cell before the first run.

The final cell removes the deployment, including its data volume and saved configuration.
Rerunning setup creates a fresh deployment and reloads the demo documents. If you
interrupt execution, run the final cleanup cell manually. Query examples read the demo data; the setup cell writes it.

Companion guides: [execution](../docs/executor.md), [explain and profile](../docs/profiling.md).

In [ ]:
from bondi.deployment import deploy_mongodb

deployment = deploy_mongodb(name="notebook-workload", port=27121)
deployment.load_demo_data()

In [ ]:
from copy import deepcopy
from dataclasses import asdict
from pprint import pprint

from bondi.mongodb import (
    MongoAggregateQuery,
    MongoCollectionRef,
    MongoExecutor,
    MongoFindQuery,
    summarize,
)
from bondi.workload import Workload, WorkloadEntry

TIMEOUT_SECONDS = 15
business = MongoCollectionRef("demo", "business")

## 1. Describe queries before executing them

A query object is data, not a connection. This workload combines three find queries
and one aggregation against the same MongoDB backend. Pattern IDs are omitted
because these examples were written by hand.

In [ ]:
summary_fields = {"_id": 0, "name": 1, "city": 1, "stars": 1, "review_count": 1}

city_query = MongoFindQuery(
    business,
    {"city": "Sydney"},
    projection=summary_fields,
    limit=20,
)
popular_query = MongoFindQuery(
    business,
    {"stars": {"$gte": 4.0}, "review_count": {"$gte": 100}},
    projection=summary_fields,
    sort=(("review_count", -1),),
    limit=20,
)
restaurant_query = MongoFindQuery(
    business,
    {"city": "Brisbane", "categories": "Restaurant"},
    projection=summary_fields,
    sort=(("stars", -1), ("review_count", -1)),
    limit=20,
)
city_totals = MongoAggregateQuery(
    business,
    [
        {"$match": {"stars": {"$gte": 3}}},
        {
            "$group": {
                "_id": "$city",
                "business_count": {"$sum": 1},
                "average_stars": {"$avg": "$stars"},
            }
        },
        {"$sort": {"business_count": -1, "_id": 1}},
        {"$limit": 10},
    ],
)
labels = ["Sydney", "Popular businesses", "Brisbane restaurants", "City totals"]
workload = Workload(
    [
        WorkloadEntry(city_query),
        WorkloadEntry(popular_query),
        WorkloadEntry(restaurant_query),
        WorkloadEntry(city_totals),
    ]
)
workload.validate()
print(f"{len(workload)} valid query descriptions; none executed yet.")

## 2. Run one query: count only or retain documents

Both modes consume the **entire cursor**, including later batches. With collection
disabled, `documents` is `None`, not an empty list. Single-query failures raise.
The context manager closes the client created by `from_uri()`.

In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=TIMEOUT_SECONDS) as executor:
    counted = executor.run(city_query)
    collected = executor.run(city_query, collect_results=True)

print(f"Count-only: {counted.row_count} rows; documents={counted.documents}")
print(f"Collected:  {collected.row_count} rows; showing the first 3")
pprint(collected.documents[:3], sort_dicts=False)
assert counted.documents is None
assert len(collected.documents) == collected.row_count

## 3. Run the workload sequentially or concurrently

`concurrency=1` runs entries in order. Larger values bound in-flight queries;
returned results still align with the original entries, including duplicates.
The small helper below displays outcomes without dumping every document.

In [ ]:
def show_results(names, results):
    for name, result in zip(names, results):
        if result is None:
            print(name, "not started")
        else:
            print(
                name,
                result.status,
                "rows:",
                result.row_count,
                "client ms:",
                round(result.elapsed_ms, 2),
            )


for concurrency in (1, 4):
    with MongoExecutor.from_uri(
        deployment.uri,
        concurrency=concurrency,
        continue_on_error=True,
        timeout_seconds=TIMEOUT_SECONDS,
    ) as executor:
        results = executor.run_workload(workload)
    print("Concurrency:", concurrency)
    show_results(labels, results)

These durations include client/network work and full cursor consumption. This is
an API demonstration, **not a controlled benchmark**: connection startup, cache
warming, run order and contention differ. Do not interpret these two runs as a
reliable speedup comparison.

## 4. Explain versus profile one query

`explain()` requests planning information. `profile()` also performs server
execution to collect statistics. Both use MongoDB explain commands and do not
return the query's documents; diagnostic timing is not normal query latency.
MongoDB explain bypasses the plan cache.

The executor preserves the raw response. `summarize()` extracts common
metrics without another database call. Planner-only metrics are `None`, not zero.
Plan stages describe the winning query plan, excluding pipeline operators.

In [ ]:
with MongoExecutor.from_uri(deployment.uri, timeout_seconds=TIMEOUT_SECONDS) as executor:
    plan = executor.explain(popular_query)
    measured_plan = executor.profile(popular_query)

print("Planner-only summary:")
pprint(asdict(summarize(plan.raw)))
print("\nMeasured summary:")
pprint(asdict(summarize(measured_plan.raw)))
print(f"Client-observed explain request: {measured_plan.elapsed_ms:.2f} ms")
assert measured_plan.documents is None

# The complete source is still available for project-specific analysis.
print("\nRaw winning plan:")
pprint(plan.raw.get("queryPlanner", {}).get("winningPlan", {}), depth=4)

## 5. Explain the entire workload

The same concurrency and error policy apply. Here we request plans only and show
each winning plan's stage names. Inspect `explanations[i].raw` for a full response. Aggregation summaries can
have unavailable fields when the parser cannot establish whole-pipeline metrics.

In [ ]:
with MongoExecutor.from_uri(
    deployment.uri,
    concurrency=2,
    continue_on_error=True,
    timeout_seconds=TIMEOUT_SECONDS,
) as executor:
    explanations = executor.explain_workload(workload)

show_results(labels, explanations)
for name, result in zip(labels, explanations):
    if result is not None and result.raw is not None:
        print(f"{name}: {summarize(result.raw).plan_stages}")

## 6. Make a failure explicit

Structural validation does not recognize every MongoDB operator. This deliberately
invalid stage passes local validation but is rejected by the server. We use
`concurrency=1` so stopping behaviour is deterministic.

A result's `error` retains the exception; `status` distinguishes `error` from
`timeout`. `None` means an entry never started. When concurrency is higher,
already-running queries finish even after scheduling stops.

In [ ]:
invalid_query = MongoAggregateQuery(business, [{"$unknown": True}])
error_workload = Workload(
    [
        WorkloadEntry(city_query),
        WorkloadEntry(invalid_query),
        WorkloadEntry(restaurant_query),
    ]
)

for continue_on_error in (False, True):
    with MongoExecutor.from_uri(
        deployment.uri,
        concurrency=1,
        continue_on_error=continue_on_error,
        timeout_seconds=TIMEOUT_SECONDS,
    ) as executor:
        outcomes = executor.run_workload(error_workload)
    print("Continue on error:", continue_on_error)
    show_results(["Valid find", "Deliberately invalid stage", "Another valid find"], outcomes)
    for result in outcomes:
        if result is not None and result.error is not None:
            print(type(result.error).__name__, str(result.error)[:180])
    assert outcomes[1].status == "error"
    if not continue_on_error:
        assert outcomes[2] is None

## 7. Save a query and make an independent variant

Workload entries retain query references. Use `deepcopy()` before editing when you
want a separate variant. Queries and workloads can both be restored. The workload example uses a temporary
directory that is automatically removed afterward.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

variant = deepcopy(city_query)
variant.limit = 5
restored = MongoFindQuery.from_json(variant.to_json())

assert city_query.limit == 20
assert restored.fingerprint() == variant.fingerprint()
assert variant.fingerprint() != city_query.fingerprint()
print("Original limit:", city_query.limit, "| Variant limit:", variant.limit)
print("Variant fingerprint:", variant.fingerprint()[:16], "…")


with TemporaryDirectory() as folder:
    path = Path(folder) / "workload.jsonl"
    workload.save(path)
    restored_workload = Workload.load(path)
    assert [e.query.fingerprint() for e in restored_workload] == [
        e.query.fingerprint() for e in workload
    ]
    print(f"Saved and restored {len(restored_workload)} workload entries")

### Try next

- Change a city or threshold in an independent query variant, then run and explain it.
- Inspect `explanations[3].raw` to compare aggregation and find response shapes.
- Enable `collect_results=True` for `city_totals` to see its grouped rows.

Do not mutate queries while they are executing. A timeout is an incomplete attempt,
not a successful query with zero results. For API details, see the local
[executor guide](../docs/executor.md).

## Remove the deployment

Remove the database container, data volume and saved configuration. All executors
and temporary samples have already been closed by their context managers.
Run the setup cell again before continuing with database examples.


In [ ]:
deployment.remove(delete_data=True)
print("Deployment and data removed.")